# Engineer causal departure-backlog features

Extend the existing departure feature dataset with a separate, causal backlog experiment path. For each sample flight, the scheduled departure timestamp `DATE` is the prediction cutoff. The trailing cohort contains only other flights scheduled earlier within the configured window. Delay aggregates use only cohort flights that pushed back strictly before the cutoff; earlier-scheduled flights still waiting to push back contribute only to the pending count.

The existing `data/features/AIRPORT_YEAR_departures.csv` file is read without modification. Learned imputation, scaling, encoding, and feature selection remain in the later model pipeline.

In [1]:
YEAR = 2019

AIRPORT = "JFK"

BACKLOG_WINDOW_MINUTES = 30

In [2]:
# Parameters
YEAR = 2024
AIRPORT = "JFK"
BACKLOG_WINDOW_MINUTES = 60


## Configure the source and output files

The notebook works from either the project root or the `notebooks` directory. The output filename records the backlog-window length so later experiments cannot accidentally mix datasets built with different cutoffs.

In [3]:
from pathlib import Path
import sys

import pandas as pd

if (Path.cwd() / "data").is_dir() and (Path.cwd() / "notebooks").is_dir():
    PROJECT_ROOT = Path.cwd()
elif Path.cwd().name == "notebooks" and (Path.cwd().parent / "data").is_dir():
    PROJECT_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError(
        "Start this notebook from the capstone project root or its notebooks directory"
    )

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from feature_engineering import MODEL_TARGETS
from feature_engineering_backlog import (
    actual_departure_timestamps,
    add_departure_backlog_features,
    backlog_feature_names,
    validate_departure_backlog_features,
)

AIRPORT = str(AIRPORT).strip().upper()
YEAR = int(YEAR)
BACKLOG_WINDOW_MINUTES = int(BACKLOG_WINDOW_MINUTES)
INPUT_FILE = PROJECT_ROOT / f"data/features/{AIRPORT}_{YEAR}_departures.csv"
OUTPUT_FILE = PROJECT_ROOT / (
    f"data/features/{AIRPORT}_{YEAR}_departures_backlog_w"
    f"{BACKLOG_WINDOW_MINUTES}.csv"
)
BACKLOG_FEATURES = backlog_feature_names(BACKLOG_WINDOW_MINUTES)

print(pd.Series({"input": str(INPUT_FILE), "output": str(OUTPUT_FILE)}))

input     /Users/johnkyte/Projects/berkeley_ml_and_ai/ca...
output    /Users/johnkyte/Projects/berkeley_ml_and_ai/ca...
dtype: str


## Load and validate the departure population

Validate rather than silently filter the existing feature dataset. The backlog reconstruction requires complete scheduled timestamps and completed-flight departure outcomes.

In [4]:
if not INPUT_FILE.is_file():
    raise FileNotFoundError(f"Departure feature file does not exist: {INPUT_FILE}")

source = pd.read_csv(INPUT_FILE, low_memory=False)
required_columns = {
    "Year", "Origin", "DATE", "DepDelay", "DepDelayMinutes",
    "DepDel15", MODEL_TARGETS["1A"],
}
missing_columns = required_columns - set(source.columns)
if missing_columns:
    raise KeyError(f"Departure feature data is missing: {sorted(missing_columns)}")

origin = source["Origin"].astype("string").str.strip().str.upper()
source_year = pd.to_numeric(source["Year"], errors="coerce")
if not origin.eq(AIRPORT).all():
    raise ValueError(f"Departure input contains origins other than {AIRPORT}")
if not source_year.eq(YEAR).all():
    raise ValueError(f"Departure input contains years other than {YEAR}")
if set(BACKLOG_FEATURES) & set(source.columns):
    raise ValueError("Input already contains backlog features; use the base departure file")

target = pd.to_numeric(source[MODEL_TARGETS["1A"]], errors="coerce")
if target.isna().any() or not target.isin([0, 1]).all():
    raise ValueError("DepDel15 must be complete and binary")

print(pd.Series({
    "rows": len(source),
    "columns": len(source.columns),
    "delayed": int(target.sum()),
}))

rows       104715
columns       112
delayed     21344
dtype: int64


## Add the trailing backlog features

`PENDING_COUNT` is the number of earlier-scheduled flights in the window that had not pushed back by the sample cutoff. Delay counts and delay summaries use only flights whose gate-out event was already observable. Empty-history rates and means remain missing so later model pipelines can learn their treatment from training data.

In [5]:
features = add_departure_backlog_features(
    source, window_minutes=BACKLOG_WINDOW_MINUTES
)
if len(features) != len(source):
    raise ValueError("Backlog feature engineering changed the departure row count")
if not features[source.columns].equals(source):
    raise ValueError("Backlog feature engineering changed one or more source columns")

feature_validation = validate_departure_backlog_features(
    features, window_minutes=BACKLOG_WINDOW_MINUTES
)

print(pd.Series({
    "backlog features": len(BACKLOG_FEATURES),
    "rows with pending backlog": int((features[BACKLOG_FEATURES[2]] > 0).sum()),
    "rows without completed history": int(features[BACKLOG_FEATURES[1]].eq(0).sum()),
}))

backlog features                      8
rows with pending backlog         84692
rows without completed history      680
dtype: int64


In [6]:
feature_validation

,dtype,missing_count,missing_percent,minimum,maximum
BACKLOG_W60_SCHEDULED_COUNT,Int32,0,0.000000,0.0,35.0
BACKLOG_W60_COMPLETED_COUNT,Int32,0,0.000000,0.0,34.0
BACKLOG_W60_PENDING_COUNT,Int32,0,0.000000,0.0,20.0
BACKLOG_W60_DELAYED_DEPARTURE_COUNT,Int32,0,0.000000,0.0,10.0
BACKLOG_W60_DELAY_RATE,float64,680,0.649382,0.0,1.0
BACKLOG_W60_MEAN_DEP_DELAY,float64,680,0.649382,-22.0,59.0
BACKLOG_W60_MEAN_DEP_DELAY_MINUTES,float64,680,0.649382,0.0,59.0
BACKLOG_W60_TOTAL_DEP_DELAY_MINUTES,float64,0,0.000000,0.0,325.0


## Validate the prediction cutoff

All flights with the same scheduled timestamp must receive the same backlog state. This confirms that no sample outcome—or outcome from another flight scheduled at the same time—enters the trailing cohort. The reconstructed actual timestamp is used only to decide whether an earlier-scheduled flight had already pushed back.

In [7]:
scheduled_timestamp = pd.to_datetime(features["DATE"], errors="coerce")
actual_timestamp = actual_departure_timestamps(features)
if scheduled_timestamp.isna().any() or actual_timestamp.isna().any():
    raise ValueError("Backlog timing audit found an invalid timestamp")

same_cutoff_variants = (
    features.assign(_CUTOFF=scheduled_timestamp)
    .groupby("_CUTOFF", sort=False)[BACKLOG_FEATURES]
    .nunique(dropna=False)
)
if same_cutoff_variants.to_numpy().max(initial=0) > 1:
    raise ValueError("Flights at the same cutoff received different backlog features")

timing_audit = pd.Series({
    "scheduled cutoffs": int(scheduled_timestamp.nunique()),
    "rows sharing a cutoff": int(scheduled_timestamp.duplicated(keep=False).sum()),
    "actual departures before their own scheduled time": int(
        (actual_timestamp < scheduled_timestamp).sum()
    ),
    "same-cutoff backlog inconsistencies": 0,
}, name="backlog timing validation")
timing_audit

scheduled cutoffs                                    60924
rows sharing a cutoff                                65192
actual departures before their own scheduled time    63850
same-cutoff backlog inconsistencies                      0
Name: backlog timing validation, dtype: int64

## Save the backlog-enhanced departure dataset

Retain every source and audit column, append the new backlog fields, and write a separate dataset. Existing feature files remain intact.

In [8]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
features.to_csv(OUTPUT_FILE, index=False)

summary = pd.Series({
    "airport": AIRPORT,
    "year": YEAR,
    "window minutes": BACKLOG_WINDOW_MINUTES,
    "rows": len(features),
    "columns": len(features.columns),
    "added columns": len(BACKLOG_FEATURES),
    "target": MODEL_TARGETS["1A"],
    "output": str(OUTPUT_FILE),
}, name="departure backlog feature summary")
print(f"Saved {len(features):,} rows to {OUTPUT_FILE}")
summary

Saved 104,715 rows to /Users/johnkyte/Projects/berkeley_ml_and_ai/capstone/data/features/JFK_2024_departures_backlog_w60.csv


airport                                                         JFK
year                                                           2024
window minutes                                                   60
rows                                                         104715
columns                                                         120
added columns                                                     8
target                                                     DepDel15
output            /Users/johnkyte/Projects/berkeley_ml_and_ai/ca...
Name: departure backlog feature summary, dtype: object